In [1]:
from pathlib import Path
from tkinter import BOTTOM, Button, Label, filedialog
import tkinter as tk
from PIL import Image, ImageTk
from keras.models import load_model
import numpy as np

/Users/s932172@aics.espritscholen.nl/bvenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def find_model_path():
    roots = []
    current = Path.cwd().resolve()
    roots.extend([current, current.parent])

    for root in list(roots):
        parent = root.parent
        if parent != root:
            roots.append(parent)

    seen = set()
    for root in roots:
        if root in seen:
            continue
        seen.add(root)
        candidate = next(sorted(root.glob('**/traffic_classifier.h5')), None)
        if candidate is not None:
            return candidate
    return current/'traffic_classifier.h5'

# MODEL_PATH = find_model_path()
MODEL_PATH = "/Users/s932172@aics.espritscholen.nl/Desktop/game development/deep_learning/lesson_8/traffic_classifier.h5"

# if not MODEL_PATH.exists():
#     raise FileNotFoundError(
#         f"Model file not found: {MODEL_PATH}"
#         "Train the model first by running the training script in the Lesson 8 folder "
#         "or place traffic_classifier.h5 in the current project folder."
#     )


model = load_model(str(MODEL_PATH))

In [3]:
classes = {
    0: 'Speed limit (20km/h)',

    1: 'Speed limit (30km/h)',

    2: 'Speed limit (50km/h)',

    3: 'Speed limit (60km/h)',

    4: 'Speed limit (70km/h)',

    5: 'Speed limit (80km/h)',

    6: 'End of speed limit (80km/h)',

    7: 'Speed limit (100km/h)',

    8: 'Speed limit (120km/h)',

    9: 'No passing',

    10: 'No passing veh over 3.5 tons',

    11: 'Right-of-way at intersection',

    12: 'Priority road',

    13: 'Yield',

    14: 'Stop',

    15: 'No vehicles',

    16: 'Veh > 3.5 tons prohibited',

    17: 'No entry',

    18: 'General caution',

    19: 'Dangerous curve left',

    20: 'Dangerous curve right',

    21: 'Double curve',

    22: 'Bumpy road',

    23: 'Slippery road',

    24: 'Road narrows on the right',

    25: 'Road work',

    26: 'Traffic signals',

    27: 'Pedestrians',

    28: 'Children crossing',

    29: 'Bicycles crossing',

    30: 'Beware of ice/snow',

    31: 'Wild animals crossing',

    32: 'End speed + passing limits',

    33: 'Turn right ahead',

    34: 'Turn left ahead',

    35: 'Ahead only',

    36: 'Go straight or right',

    37: 'Go straight or left',

    38: 'Keep right',

    39: 'Keep left',

    40: 'Roundabout mandatory',

    41: 'End of no passing',

    42: 'End no passing veh > 3.5 tons'
}


In [4]:
top = tk.Tk()
top.geometry('800x600')
top.title('Traffic Sign Classification')
top.configure(background='#CDCDCD')

label = Label(top, background='#CDCDCD', font=('arial', 15, 'bold'))
label.place(x=10, y=10)
sign_image = Label(top)
classify_button = None

In [5]:
def process_image(image):
    image = image.convert('RGB').resize((32,32))
    image = np.asarray(image, dtype=np.float32) / 255.0
    return np.expand_dims(image, axis=0)

def classify(file_path):
    try:
        image = Image.open(file_path)
        image_array = process_image(image)
        pred = model.predict(image_array, verbose=0)
        predicted_class = int(np.argmax(pred, axis=1)[0])
        sign = classes.get(predicted_class, f'Unknown class {predicted_class}')
        print(f'Predicted Class: {predicted_class}, Label: {sign}')
        label.configure(foreground='#011638', text=sign)
    except Exception as exc:
        label.configure(foreground='red', text=f'Error: {exc}')
        print(f'Error classifying image: {exc}')

def show_classify_button(file_path):
    global classify_button
    if classify_button is not None:
        classify_button.destroy()
    
    classify_button = Button(
        top,
        text='Classify Image',
        command=lambda: classify(file_path),
        padx=10,
        pady=5
    )
    classify_button.configure(background='#364156', foreground='black', font=('arial', 10, 'bold'))
    classify_button.place(relx=0.79, rely=0.46)


def upload_image():
    try:
        file_path = filedialog.askopenfilename()
        if not file_path:
            return
        
        uploaded = Image.open(file_path).convert('RGB')
        width = max(1, int(top.winfo_width() / 2.25))
        height = max(1, int(top.winfo_height() / 2.25))
        uploaded.thumbnail((width, height))
        im = ImageTk.PhotoImage(uploaded)
        sign_image.configure(image=im)
        sign_image.image = im
        label.configure(text='')
        show_classify_button(file_path)
    except Exception as exc:
        print(f'Error uploading image: {exc}')
        label.configure(foreground='red', text='Please select a valid image file.')


upload = Button(
    top,
    text='upload an image',
    command=upload_image,
    padx=10,
    pady=5
)
upload.configure(background='#364156', foreground='black', font=('arial', 10, 'bold'))
upload.pack(side=BOTTOM, pady=50)

sign_image.pack(side=BOTTOM, expand=True)
label.pack(side=BOTTOM, expand=True)

heading = Label(top, text='Know Your Traffic Sign', pady=20, font=('arial', 20, 'bold'))
heading.configure(background='#CDCDCD', foreground='#364156')
heading.pack()

top.mainloop()

Predicted Class: 4, Label: Speed limit (70km/h)
Predicted Class: 25, Label: Road work
Predicted Class: 9, Label: No passing
Predicted Class: 1, Label: Speed limit (30km/h)
Predicted Class: 17, Label: No entry
